In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
import math
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)

# 检查CUDA状态（方案2）
try:
    # 尝试使用CUDA
    device = torch.device("cuda:0")
    # 测试CUDA是否真的可用
    torch.randn(1).to(device)
    print(f"使用CUDA: {device}")
    print(f"CUDA版本: {torch.version.cuda}")
    print(f"GPU名称: {torch.cuda.get_device_name(0)}")
except Exception as e:
    # 如果CUDA不可用，使用CPU
    print(f"CUDA错误: {e}")
    print("使用CPU训练")
    device = torch.device("cpu")

print(f"最终使用设备: {device}")

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)

d:\anaconda\envs\transformer_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sys.version_info(major=3, minor=12, micro=12, releaselevel='final', serial=0)
matplotlib 3.10.8
numpy 2.3.5
pandas 3.0.1
sklearn 1.8.0
torch 2.10.0+cu128
使用CUDA: cuda:0
CUDA版本: 12.8
GPU名称: NVIDIA GeForce RTX 5070 Laptop GPU
最终使用设备: cuda:0


## 数据加载

- 采用中文-英语平行语料库

In [2]:
# 数据加载函数（优化版）
def load_data(data_dir, mode, max_samples=None):
    """加载预处理后的数据，支持分批加载"""
    src_file = os.path.join(data_dir, f'{mode}_src.bpe')
    trg_file = os.path.join(data_dir, f'{mode}_trg.bpe')
    
    src_lines = []
    trg_lines = []
    
    print(f"加载 {src_file}...")
    with open(src_file, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_samples is not None and i >= max_samples:
                break
            if line.strip():
                src_lines.append(line.strip().split())
    
    print(f"加载 {trg_file}...")
    with open(trg_file, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_samples is not None and i >= max_samples:
                break
            if line.strip():
                trg_lines.append(line.strip().split())
    
    return src_lines, trg_lines


def load_data_range(data_dir, mode, start_idx, end_idx):
    """加载指定范围的数据，避免读取整个文件
    
    Args:
        data_dir: 数据目录
        mode: 数据模式（train/val/test）
        start_idx: 起始行索引（包含）
        end_idx: 结束行索引（不包含）
    
    Returns:
        src_lines: 源语言数据
        trg_lines: 目标语言数据
    """
    src_file = os.path.join(data_dir, f'{mode}_src.bpe')
    trg_file = os.path.join(data_dir, f'{mode}_trg.bpe')
    
    src_lines = []
    trg_lines = []
    
    print(f"加载 {src_file} (行 {start_idx}-{end_idx})...")
    
    # 跳过前面的行，只读取指定范围
    with open(src_file, 'r', encoding='utf-8') as f:
        # 跳过start_idx行
        for _ in range(start_idx):
            next(f)
        
        # 读取end_idx - start_idx行
        for _ in range(end_idx - start_idx):
            try:
                line = next(f)
                if line.strip():
                    src_lines.append(line.strip().split())
            except StopIteration:
                break
    
    # 同样处理目标文件
    with open(trg_file, 'r', encoding='utf-8') as f:
        # 跳过start_idx行
        for _ in range(start_idx):
            next(f)
        
        # 读取end_idx - start_idx行
        for _ in range(end_idx - start_idx):
            try:
                line = next(f)
                if line.strip():
                    trg_lines.append(line.strip().split())
            except StopIteration:
                break
    
    print(f"加载完成，共 {len(src_lines)} 行")
    return src_lines, trg_lines

# 加载词汇表
def load_vocab(vocab_file):
    """加载词汇表"""
    vocab = {}
    with open(vocab_file, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            word = line.strip()
            if word:
                vocab[word] = idx
    return vocab

# 生成词汇表
def generate_vocab(src_lines, trg_lines, min_freq=2):
    """从数据中生成词汇表"""
    from collections import Counter
    
    # 统计词频
    word_counts = Counter()
    
    # 统计源语言和目标语言的词频
    for line in src_lines:
        word_counts.update(line)
    for line in trg_lines:
        word_counts.update(line)
    
    # 生成词汇表，只包含频率大于等于min_freq的词
    vocab = {}
    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = len(vocab)
    
    return vocab

# 配置数据路径
data_dir = 'wmt16'
vocab_file = os.path.join(data_dir, 'vocab')

# 分批训练配置
# 训练集总句子数: 22,277,119
# 每批训练样本数: 1,000,000
# 批次数: 23 (确保覆盖所有句子)

batch_train_samples = 1000000  # 每批训练的样本数
total_train_samples = None  # 总训练样本数，None表示使用全部数据
model_save_dir = 'checkpoints'  # 模型保存目录
os.makedirs(model_save_dir, exist_ok=True)

# 加载数据（先加载验证和测试数据）
print("加载数据...")
val_src, val_trg = load_data(data_dir, 'val')
test_src, test_trg = load_data(data_dir, 'test')

# 加载或生成词汇表
if os.path.exists(vocab_file) and os.path.getsize(vocab_file) > 0:
    # 加载已存在的非空词汇表
    vocab = load_vocab(vocab_file)
    print(f'加载的词汇表大小: {len(vocab)}')
else:
    # 词汇表文件不存在或为空，从训练数据生成
    print('词汇表文件不存在或为空，正在从训练数据生成...')
    # 加载一部分训练数据用于生成词汇表
    temp_train_src, temp_train_trg = load_data(data_dir, 'train', max_samples=20000)
    # 生成词汇表
    vocab = generate_vocab(temp_train_src, temp_train_trg)
    print(f'生成的词汇表大小: {len(vocab)}')
    
    # 保存词汇表到文件
    os.makedirs(os.path.dirname(vocab_file), exist_ok=True)
    with open(vocab_file, 'w', encoding='utf-8') as f:
        for word in vocab:
            f.write(word + '')
    print(f'词汇表已保存到: {vocab_file}')

vocab_size = len(vocab)

print(f'验证数据: {len(val_src)} 句')
print(f'测试数据: {len(test_src)} 句')
print(f'词汇表大小: {vocab_size}')
print(f'每批训练样本数: {batch_train_samples}')

加载数据...
加载 wmt16\val_src.bpe...
加载 wmt16\val_trg.bpe...
加载 wmt16\test_src.bpe...
加载 wmt16\test_trg.bpe...
加载的词汇表大小: 12953
验证数据: 1237618 句
测试数据: 1237618 句
词汇表大小: 12953
每批训练样本数: 1000000


## 数据预处理

In [3]:
# 特殊标记
PAD_TOKEN = '<pad>'
SOS_TOKEN = '<sos>'
EOS_TOKEN = '<eos>'
UNK_TOKEN = '<unk>'

# 添加特殊标记到词汇表
special_tokens = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
for token in special_tokens:
    if token not in vocab:
        vocab[token] = len(vocab)

# 反向词汇表
reverse_vocab = {v: k for k, v in vocab.items()}

# 最大序列长度
max_seq_len = 50

# 预计算特殊标记的索引
sos_idx = vocab.get(SOS_TOKEN, 0)
eos_idx = vocab.get(EOS_TOKEN, 1)
pad_idx = vocab.get(PAD_TOKEN, 2)
unk_idx = vocab.get(UNK_TOKEN, 3)

def tokenize_and_pad(lines, vocab, max_len):
    """将文本转换为索引并进行填充（优化版）"""
    import time
    start_time = time.time()
    
    batch_size = len(lines)
    # 初始化结果张量，形状为 [batch_size, max_len]
    result = torch.full((batch_size, max_len), pad_idx, dtype=torch.long)
    
    for i, line in enumerate(lines):
        # 转换为索引，使用列表推导式快速处理
        tokens = [vocab.get(word, unk_idx) for word in line]
        
        # 计算实际长度（加上SOS和EOS）
        actual_len = len(tokens) + 2
        
        # 确定有效长度（不超过max_len）
        valid_len = min(actual_len, max_len)
        
        if valid_len > 0:
            # 填充SOS标记
            result[i, 0] = sos_idx
            
            # 填充实际词索引（如果有空间）
            if valid_len > 1:
                content_len = min(len(tokens), valid_len - 1)
                result[i, 1:1+content_len] = torch.tensor(tokens[:content_len], dtype=torch.long)
                
                # 填充EOS标记（如果有空间）
                if valid_len > content_len + 1:
                    result[i, 1+content_len] = eos_idx
    
    end_time = time.time()
    print(f"Tokenize and pad took {end_time - start_time:.4f} seconds for {batch_size} samples")
    
    return result

# 预处理数据（只预处理验证和测试数据）
print("预处理数据...")
val_src_tensor = tokenize_and_pad(val_src, vocab, max_seq_len)
val_trg_tensor = tokenize_and_pad(val_trg, vocab, max_seq_len)
test_src_tensor = tokenize_and_pad(test_src, vocab, max_seq_len)
test_trg_tensor = tokenize_and_pad(test_trg, vocab, max_seq_len)

print(f'验证数据形状: 源语言 {val_src_tensor.shape}, 目标语言 {val_trg_tensor.shape}')
print(f'测试数据形状: 源语言 {test_src_tensor.shape}, 目标语言 {test_trg_tensor.shape}')

预处理数据...


Tokenize and pad took 61.0015 seconds for 1237618 samples
Tokenize and pad took 38.1439 seconds for 1237618 samples
Tokenize and pad took 42.8715 seconds for 1237618 samples
Tokenize and pad took 44.3657 seconds for 1237618 samples
验证数据形状: 源语言 torch.Size([1237618, 50]), 目标语言 torch.Size([1237618, 50])
测试数据形状: 源语言 torch.Size([1237618, 50]), 目标语言 torch.Size([1237618, 50])


## Transformer模型定义

In [4]:
# 位置编码
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.max_len = max_len
        self.d_model = d_model
        
        # 使用Embedding层实现位置编码
        self.pos_embedding = nn.Embedding(
            self.max_len,
            self.d_model,
            _weight=self.get_positional_encoding(self.max_len, self.d_model)
        )
        self.pos_embedding.weight.requires_grad_(False)  # 不更新位置编码的权重

    def get_positional_encoding(self, max_length, hidden_size):
        """计算位置编码"""
        pe = torch.zeros(max_length, hidden_size)
        position = torch.arange(0, max_length).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, hidden_size, 2)
            * -(torch.log(torch.Tensor([10000.0])) / hidden_size)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, x):
        """前向传播"""
        # x: [batch_size, seq_len, d_model]
        batch_size, seq_len, _ = x.size()
        
        # 生成位置 IDs
        position_ids = torch.arange(seq_len, dtype=torch.long, device=x.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
        
        # 获取位置编码
        pos_embeds = self.pos_embedding(position_ids)
        
        # 与输入相加
        x = x + pos_embeds
        return x

# 注意力机制
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, nhead):
        super(MultiHeadAttention, self).__init__()
        self.nhead = nhead
        self.d_model = d_model
        self.d_k = d_model // nhead
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)
    
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        q_seq_len = q.size(1)
        k_seq_len = k.size(1)
        
        # 线性变换并分多头
        q = self.q_linear(q).reshape(batch_size, q_seq_len, self.nhead, self.d_k).transpose(1, 2)
        k = self.k_linear(k).reshape(batch_size, k_seq_len, self.nhead, self.d_k).transpose(1, 2)
        v = self.v_linear(v).reshape(batch_size, k_seq_len, self.nhead, self.d_k).transpose(1, 2)
        
        # 计算注意力权重
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            # 确保掩码形状正确
            if mask.dim() == 4:
                mask = mask.squeeze(1)
            mask = mask.unsqueeze(1)
            attn_weights = attn_weights.masked_fill(mask == 0, -1e4)  # 使用在半精度范围内的值
        
        attn_weights = F.softmax(attn_weights, dim=-1)
        
        # 应用注意力
        output = torch.matmul(attn_weights, v)
        output = output.transpose(1, 2).contiguous().reshape(batch_size, q_seq_len, self.d_model)
        output = self.out_linear(output)
        
        return output, attn_weights

# 前馈网络
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = self.dropout(x)
        x = self.linear2(x)
        return x

# 编码器层
class EncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, d_ff):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, src_mask):
        # 自注意力
        attn_output, _ = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # 前馈网络
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

# 解码器层
class DecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, d_ff):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead)
        self.cross_attn = MultiHeadAttention(d_model, nhead)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, enc_output, tgt_mask, src_tgt_mask):
        # 自注意力
        attn_output, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # 交叉注意力
        attn_output, _ = self.cross_attn(x, enc_output, enc_output, src_tgt_mask)
        x = self.norm2(x + self.dropout(attn_output))
        
        # 前馈网络
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        
        return x

# 编码器
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, d_ff):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, nhead, d_ff) for _ in range(num_layers)])
        self.dropout = nn.Dropout(0.1)

    def forward(self, src, src_mask):
        x = self.embedding(src)
        x = self.pos_encoder(x)
        x = self.dropout(x)
        
        for layer in self.layers:
            x = layer(x, src_mask)
        
        return x

# 解码器
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, d_ff):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([DecoderLayer(d_model, nhead, d_ff) for _ in range(num_layers)])
        self.dropout = nn.Dropout(0.1)

    def forward(self, tgt, enc_output, tgt_mask, src_tgt_mask):
        x = self.embedding(tgt)
        x = self.pos_encoder(x)
        x = self.dropout(x)
        
        for layer in self.layers:
            x = layer(x, enc_output, tgt_mask, src_tgt_mask)
        
        return x

# Transformer模型
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, nhead, num_layers, d_ff):
        super(Transformer, self).__init__()
        self.encoder = Encoder(src_vocab_size, d_model, nhead, num_layers, d_ff)
        self.decoder = Decoder(tgt_vocab_size, d_model, nhead, num_layers, d_ff)
        self.generator = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask, tgt_mask, src_tgt_mask):
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(tgt, enc_output, tgt_mask, src_tgt_mask)
        output = self.generator(dec_output)
        return output

## 模型初始化

In [5]:
# 模型参数（进一步增大以提升模型容量）
d_model = 512
nhead = 8
num_layers = 6
d_ff = 2048
src_vocab_size = len(vocab)
tgt_vocab_size = len(vocab)

# 创建模型
print("创建模型...")
model = Transformer(src_vocab_size, tgt_vocab_size, d_model, nhead, num_layers, d_ff)
model = model.to(device)

# 优化器
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
# 学习率调度器
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
# 损失函数
criterion = nn.CrossEntropyLoss(ignore_index=vocab[PAD_TOKEN])

print(f'模型参数: {sum(p.numel() for p in model.parameters()):,}')

# 模型保存和加载函数
def save_checkpoint(model, optimizer, scheduler, epoch, loss, accuracy, checkpoint_path):
    """保存模型检查点"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss,
        'accuracy': accuracy,
    }
    torch.save(checkpoint, checkpoint_path)
    print(f'模型已保存到: {checkpoint_path}')

def load_checkpoint(model, optimizer, scheduler, checkpoint_path):
    """加载模型检查点"""
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    accuracy = checkpoint['accuracy']
    print(f'模型已加载，epoch: {epoch}, loss: {loss:.4f}, accuracy: {accuracy:.4f}')
    return epoch, loss, accuracy

创建模型...
模型参数: 69,173,405


## 训练函数

In [6]:
# 生成掩码
def generate_masks(src, tgt, pad_idx):
    """生成注意力掩码"""
    batch_size, src_len = src.size()
    _, tgt_len = tgt.size()
    
    # 源语言掩码
    src_mask = (src != pad_idx).unsqueeze(1).unsqueeze(2)
    
    # 目标语言掩码（防止看到未来的词）
    tgt_mask = (tgt != pad_idx).unsqueeze(1).unsqueeze(3)
    tgt_sub_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=device)).bool()
    tgt_mask = tgt_mask & tgt_sub_mask
    
    # 源语言-目标语言掩码
    src_tgt_mask = (src != pad_idx).unsqueeze(1).unsqueeze(2)
    
    return src_mask, tgt_mask, src_tgt_mask

# 训练函数
def train_epoch(model, train_src, train_trg, optimizer, criterion, pad_idx, batch_size=32):
    """训练一个epoch（带实时进度显示）"""
    model.train()
    total_loss = 0
    total_correct = 0
    total_tokens = 0
    
    num_batches = len(train_src) // batch_size
    
    # 创建进度条，显示实时信息
    pbar = tqdm(range(0, len(train_src), batch_size), desc='Training')
    
    for i in pbar:
        end = i + batch_size
        if end > len(train_src):
            break
        
        src_batch = train_src[i:end].to(device)
        trg_batch = train_trg[i:end].to(device)
        
        # 目标输入（去掉EOS）
        trg_input = trg_batch[:, :-1]
        # 目标输出（去掉SOS）
        trg_output = trg_batch[:, 1:]
        
        # 生成掩码
        src_mask, tgt_mask, src_tgt_mask = generate_masks(src_batch, trg_input, pad_idx)
        
        # 前向传播
        optimizer.zero_grad()
        output = model(src_batch, trg_input, src_mask, tgt_mask, src_tgt_mask)
        
        # 计算损失
        loss = criterion(output.view(-1, output.size(-1)), trg_output.reshape(-1))
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        # 计算准确率
        pred = output.argmax(dim=-1)
        correct = (pred == trg_output).sum().item()
        tokens = (trg_output != pad_idx).sum().item()
        
        total_loss += loss.item()
        total_correct += correct
        total_tokens += tokens
        
        # 更新进度条显示实时信息
        current_loss = total_loss / ((i // batch_size) + 1)
        current_acc = total_correct / total_tokens if total_tokens > 0 else 0
        pbar.set_postfix({
            'loss': f'{current_loss:.4f}',
            'acc': f'{current_acc:.4f}'
        })
    
    avg_loss = total_loss / num_batches
    accuracy = total_correct / total_tokens
    
    return avg_loss, accuracy

# 评估函数
def evaluate(model, val_src, val_trg, criterion, pad_idx, batch_size=32):
    """评估模型"""
    model.eval()
    total_loss = 0
    total_correct = 0
    total_tokens = 0
    
    num_batches = max(1, len(val_src) // batch_size)
    
    with torch.no_grad():
        for i in tqdm(range(0, len(val_src), batch_size), desc='Evaluating'):
            end = i + batch_size
            if end > len(val_src):
                end = len(val_src)
            
            src_batch = val_src[i:end].to(device)
            trg_batch = val_trg[i:end].to(device)
            
            # 目标输入（去掉EOS）
            trg_input = trg_batch[:, :-1]
            # 目标输出（去掉SOS）
            trg_output = trg_batch[:, 1:]
            
            # 生成掩码
            src_mask, tgt_mask, src_tgt_mask = generate_masks(src_batch, trg_input, pad_idx)
            
            # 前向传播
            output = model(src_batch, trg_input, src_mask, tgt_mask, src_tgt_mask)
            
            # 计算损失
            loss = criterion(output.view(-1, output.size(-1)), trg_output.reshape(-1))
            
            # 计算准确率
            pred = output.argmax(dim=-1)
            correct = (pred == trg_output).sum().item()
            tokens = (trg_output != pad_idx).sum().item()
            
            total_loss += loss.item()
            total_correct += correct
            total_tokens += tokens
    
    avg_loss = total_loss / num_batches
    accuracy = total_correct / total_tokens
    
    return avg_loss, accuracy

## 分批训练（支持模型保存和加载）

In [7]:
# 批次断点保存和加载函数
def save_batch_checkpoint(batch_idx, best_val_loss, checkpoint_dir):
    """保存批次断点信息"""
    checkpoint_file = os.path.join(checkpoint_dir, 'batch_checkpoint.txt')
    with open(checkpoint_file, 'w', encoding='utf-8') as f:
        f.write(f'{batch_idx}\n')
        f.write(f'{best_val_loss}\n')
    print(f'批次断点已保存: 批次 {batch_idx + 1}, 最佳验证损失: {best_val_loss:.4f}')

def load_batch_checkpoint(checkpoint_dir):
    """加载批次断点信息"""
    checkpoint_file = os.path.join(checkpoint_dir, 'batch_checkpoint.txt')
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            batch_idx = int(lines[0].strip())
            best_val_loss = float(lines[1].strip())
        print(f'发现批次断点: 批次 {batch_idx + 1}, 最佳验证损失: {best_val_loss:.4f}')
        return batch_idx, best_val_loss
    return None, None

# 训练参数
num_epochs = 23
batch_size = 32
pad_idx = vocab[PAD_TOKEN]

# 记录训练过程
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

# 分批训练配置
# 训练集总句子数: 22,277,119
# 每批训练样本数: 1,000,000
# 批次数: 23 (确保覆盖所有句子)

num_batches_per_round = 23  # 每轮训练的批次数量
epochs_per_batch = 1  # 每批数据训练的epoch数
checkpoint_path = os.path.join(model_save_dir, 'best_model.pt')
start_batch = 0
best_val_loss = float('inf')

# 检查是否存在批次断点
batch_checkpoint_idx, batch_checkpoint_loss = load_batch_checkpoint(model_save_dir)
if batch_checkpoint_idx is not None:
    # 从批次断点继续训练
    start_batch = batch_checkpoint_idx + 1
    best_val_loss = batch_checkpoint_loss
    print(f"从批次 {start_batch} 继续训练")

    # 同时检查是否存在已保存的模型
    if os.path.exists(checkpoint_path):
        print("发现已保存的模型，正在加载...")
        loaded_epoch, loaded_loss, loaded_acc = load_checkpoint(model, optimizer, scheduler, checkpoint_path)
        best_val_loss = loaded_loss
        print(f"模型已加载，epoch: {loaded_epoch}, loss: {loaded_loss:.4f}, accuracy: {loaded_acc:.4f}")
else:
    # 检查是否存在已保存的模型
    if os.path.exists(checkpoint_path):
        print("发现已保存的模型，正在加载...")
        loaded_epoch, loaded_loss, loaded_acc = load_checkpoint(model, optimizer, scheduler, checkpoint_path)
        best_val_loss = loaded_loss
        # 计算从哪个批次开始
        start_batch = loaded_epoch // epochs_per_batch
        print(f"从批次 {start_batch} 继续训练")

print("开始分批训练...")
for batch_idx in range(start_batch, num_batches_per_round):
    batch_start = batch_idx * batch_train_samples
    batch_end = batch_start + batch_train_samples
    
    print(f"\n{'='*60}")
    print(f"批次 {batch_idx + 1}/{num_batches_per_round}: 数据范围 {batch_start} - {batch_end}")
    print(f"{'='*60}")
    
    # 加载当前批次的训练数据
    print(f"加载训练数据...")
    train_src, train_trg = load_data_range(data_dir, 'train', batch_start, batch_end)
    
    # 预处理当前批次数据
    train_src_tensor = tokenize_and_pad(train_src, vocab, max_seq_len)
    train_trg_tensor = tokenize_and_pad(train_trg, vocab, max_seq_len)
    
    print(f"当前批次训练数据: {len(train_src)} 句")
    
    # 训练当前批次的多个epoch
    start_epoch = batch_idx * epochs_per_batch
    for epoch in range(epochs_per_batch):
        current_epoch = start_epoch + epoch
        print(f'\nEpoch [{current_epoch+1}/{num_epochs}] (批次 {batch_idx+1}, 子epoch {epoch+1}/{epochs_per_batch})')
        
        # 训练
        train_loss, train_acc = train_epoch(model, train_src_tensor, train_trg_tensor, optimizer, criterion, pad_idx, batch_size)
        
        # 评估
        val_loss, val_acc = evaluate(model, val_src_tensor, val_trg_tensor, criterion, pad_idx, batch_size)
        
        # 学习率调度
        scheduler.step()
        
        # 记录结果
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        
        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
        
        # 保存最佳模型
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(model, optimizer, scheduler, current_epoch, val_loss, val_acc, checkpoint_path)
            print(f"保存最佳模型，验证损失: {best_val_loss:.4f}")
        
        print('-' * 50)
    
    # 清理内存
    del train_src, train_trg, train_src_tensor, train_trg_tensor
    torch.cuda.empty_cache()
    print(f"批次 {batch_idx + 1} 完成，最佳验证损失: {best_val_loss:.4f}")
    
    # 保存批次断点
    save_batch_checkpoint(batch_idx, best_val_loss, model_save_dir)

print(f"\n训练完成！最佳验证损失: {best_val_loss:.4f}")


发现批次断点: 批次 19, 最佳验证损失: 1.7183
从批次 19 继续训练
发现已保存的模型，正在加载...
模型已加载，epoch: 18, loss: 1.7183, accuracy: 0.6533
模型已加载，epoch: 18, loss: 1.7183, accuracy: 0.6533
开始分批训练...

批次 20/23: 数据范围 19000000 - 20000000
加载训练数据...
加载 wmt16\train_src.bpe (行 19000000-20000000)...
加载完成，共 1000000 行
Tokenize and pad took 49.6736 seconds for 1000000 samples
Tokenize and pad took 50.2113 seconds for 1000000 samples
当前批次训练数据: 1000000 句

Epoch [20/23] (批次 20, 子epoch 1/1)


Evaluating: 100%|██████████| 38676/38676 [16:41<00:00, 38.61it/s]


Train Loss: 1.8201, Train Acc: 0.6367
Val Loss: 1.7073, Val Acc: 0.6544
模型已保存到: checkpoints\best_model.pt
保存最佳模型，验证损失: 1.7073
--------------------------------------------------
批次 20 完成，最佳验证损失: 1.7073
批次断点已保存: 批次 20, 最佳验证损失: 1.7073

批次 21/23: 数据范围 20000000 - 21000000
加载训练数据...
加载 wmt16\train_src.bpe (行 20000000-21000000)...
加载完成，共 1000000 行
Tokenize and pad took 48.1688 seconds for 1000000 samples
Tokenize and pad took 43.3109 seconds for 1000000 samples
当前批次训练数据: 1000000 句

Epoch [21/23] (批次 21, 子epoch 1/1)


Evaluating: 100%|██████████| 38676/38676 [17:01<00:00, 37.88it/s]


Train Loss: 1.7777, Train Acc: 0.6438
Val Loss: 1.6643, Val Acc: 0.6620
模型已保存到: checkpoints\best_model.pt
保存最佳模型，验证损失: 1.6643
--------------------------------------------------
批次 21 完成，最佳验证损失: 1.6643
批次断点已保存: 批次 21, 最佳验证损失: 1.6643

批次 22/23: 数据范围 21000000 - 22000000
加载训练数据...
加载 wmt16\train_src.bpe (行 21000000-22000000)...
加载完成，共 1000000 行
Tokenize and pad took 19.3487 seconds for 1000000 samples
Tokenize and pad took 17.6166 seconds for 1000000 samples
当前批次训练数据: 1000000 句

Epoch [22/23] (批次 22, 子epoch 1/1)


Evaluating: 100%|██████████| 38676/38676 [16:11<00:00, 39.82it/s]


Train Loss: 1.7646, Train Acc: 0.6461
Val Loss: 1.6550, Val Acc: 0.6634
模型已保存到: checkpoints\best_model.pt
保存最佳模型，验证损失: 1.6550
--------------------------------------------------
批次 22 完成，最佳验证损失: 1.6550
批次断点已保存: 批次 22, 最佳验证损失: 1.6550

批次 23/23: 数据范围 22000000 - 23000000
加载训练数据...
加载 wmt16\train_src.bpe (行 22000000-23000000)...
加载完成，共 277119 行
Tokenize and pad took 4.5954 seconds for 277119 samples
Tokenize and pad took 4.4868 seconds for 277119 samples
当前批次训练数据: 277119 句

Epoch [23/23] (批次 23, 子epoch 1/1)


Evaluating: 100%|██████████| 38676/38676 [17:01<00:00, 37.86it/s]


Train Loss: 1.7630, Train Acc: 0.6465
Val Loss: 1.6540, Val Acc: 0.6632
模型已保存到: checkpoints\best_model.pt
保存最佳模型，验证损失: 1.6540
--------------------------------------------------
批次 23 完成，最佳验证损失: 1.6540
批次断点已保存: 批次 23, 最佳验证损失: 1.6540

训练完成！最佳验证损失: 1.6540
